# Debrief plots — the reveal

Run this **after the game** to produce the five plots the facilitator projects during the 15-minute debrief.

## Input
- `formulation-storefront.xlsx` — the file the facilitator has been filling in during the game. Upload it to Colab before running (Files → Upload).

## Outputs — five .png plots
1. `reveal-01-tensile-round1.png` — tensile trajectories by team, round 1
2. `reveal-02-elongation-full-game.png` — elongation over both rounds
3. `reveal-03-pareto.png` — tensile vs elongation, colored by archetype
4. `reveal-04-team-summary.png` — best-observed responses per team
5. `reveal-05-true-surface.png` — the true elongation surface with each team's queried points overlaid

## Suggested narration order

Project plots in order **3 → 5 → 1 → 2 → 4** for the highest-impact reveal. See the facilitator run-of-show doc for narration lines.

## Setup — the true model

Needed for the true-surface plot (#5), which draws the ground-truth elongation surface for overlay.

In [ ]:
"""Latex rubber film formulation — physics simulator for the DOE tutorial game.

Public API: `simulate(**natural_units) -> dict`.
"""
from __future__ import annotations
import math
from dataclasses import dataclass


@dataclass(frozen=True)
class FactorRange:
    name: str
    low: float
    high: float

    def to_coded(self, x: float) -> float:
        mid = 0.5 * (self.low + self.high)
        half = 0.5 * (self.high - self.low)
        return (x - mid) / half

    def from_coded(self, x_c: float) -> float:
        mid = 0.5 * (self.low + self.high)
        half = 0.5 * (self.high - self.low)
        return mid + x_c * half


RANGES = {
    "latex_pct":  FactorRange("latex_pct",  5.0,  20.0),
    "filler_phr":        FactorRange("filler_phr",        0.0,  40.0),
    "crosslinker_phr":   FactorRange("crosslinker_phr",   0.5,   4.0),
    "plasticizer_phr":   FactorRange("plasticizer_phr",   0.0,  25.0),
    "cure_temp_c":       FactorRange("cure_temp_c",     100.0, 160.0),
}

LATEX_HARD_CAP = 20.0

TENSILE = dict(
    intercept  = 8.0,
    A          = 0.10,
    B          = 3.00,
    C          = 2.00,
    D          = -0.20,
    E          = 0.20,
    BC         = 1.50,
    noise_sd   = 0.60,
)

ELONGATION = dict(
    intercept       = 380.0,
    A_linear        =   3.0,
    E_linear        =   8.0,
    B_center        =  -0.5,
    B_curvature     = -100.0,
    C_center        =  +0.5,
    C_curvature     = -150.0,
    D_center        =  +0.5,
    D_curvature     =  -80.0,
    BC              =  -10.0,
    noise_sd        =  15.0,
)

HARDNESS = dict(
    intercept  = 55.0,
    A          =  0.5,
    B          = 12.0,
    C          =  5.0,
    D          = -3.0,
    E          =  2.0,
    BC         =  3.0,
    noise_sd   =  2.5,
)


def _tensile(A, B, C, D, E, rng):
    m = TENSILE
    y = (m["intercept"] + m["A"]*A + m["B"]*B + m["C"]*C + m["D"]*D + m["E"]*E
         + m["BC"]*B*C)
    return y + rng.gauss(0.0, m["noise_sd"])


def _elongation(A, B, C, D, E, rng):
    m = ELONGATION
    y = (m["intercept"]
         + m["A_linear"]*A
         + m["E_linear"]*E
         + m["B_curvature"]*(B - m["B_center"])**2
         + m["C_curvature"]*(C - m["C_center"])**2
         + m["D_curvature"]*(D - m["D_center"])**2
         + m["BC"]*B*C)
    return y + rng.gauss(0.0, m["noise_sd"])


def _hardness(A, B, C, D, E, rng):
    m = HARDNESS
    y = (m["intercept"] + m["A"]*A + m["B"]*B + m["C"]*C + m["D"]*D + m["E"]*E
         + m["BC"]*B*C)
    return y + rng.gauss(0.0, m["noise_sd"])


def _in_range(name, x):
    r = RANGES[name]
    if x < r.low or x > r.high:
        return f"{name}={x} outside [{r.low}, {r.high}]"
    return None


def simulate(latex_pct, filler_phr, crosslinker_phr,
             plasticizer_phr, cure_temp_c, *, rng=None):
    """Return a dict with the three responses plus feasibility flag."""
    import random
    if rng is None:
        rng = random.Random()

    for nm, x in (("latex_pct", latex_pct),
                  ("filler_phr", filler_phr),
                  ("crosslinker_phr", crosslinker_phr),
                  ("plasticizer_phr", plasticizer_phr),
                  ("cure_temp_c", cure_temp_c)):
        problem = _in_range(nm, x)
        if problem:
            return dict(tensile_mpa=math.nan, elongation_pct=math.nan,
                        hardness_shore_a=math.nan, infeasible=problem)

    if latex_pct > LATEX_HARD_CAP:
        return dict(tensile_mpa=math.nan, elongation_pct=math.nan,
                    hardness_shore_a=math.nan,
                    infeasible=f"latex > {LATEX_HARD_CAP}% — mixture unstable")

    A = RANGES["latex_pct"].to_coded(latex_pct)
    B = RANGES["filler_phr"].to_coded(filler_phr)
    C = RANGES["crosslinker_phr"].to_coded(crosslinker_phr)
    D = RANGES["plasticizer_phr"].to_coded(plasticizer_phr)
    E = RANGES["cure_temp_c"].to_coded(cure_temp_c)

    return dict(
        tensile_mpa=_tensile(A, B, C, D, E, rng),
        elongation_pct=_elongation(A, B, C, D, E, rng),
        hardness_shore_a=_hardness(A, B, C, D, E, rng),
        infeasible=None,
    )


def simulate_coded(A, B, C, D, E, *, rng=None):
    """Coded [-1,+1] factor levels -> responses."""
    return simulate(
        RANGES["latex_pct"].from_coded(A),
        RANGES["filler_phr"].from_coded(B),
        RANGES["crosslinker_phr"].from_coded(C),
        RANGES["plasticizer_phr"].from_coded(D),
        RANGES["cure_temp_c"].from_coded(E),
        rng=rng,
    )


In [ ]:
# %pip install openpyxl pandas matplotlib numpy

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


TEAM_COLORS = {
    "OFAT":  "#C0392B",
    "FRAC":  "#F39C12",
    "CCD":   "#2874A6",
    "OTHER": "#7B7D7D",
}


def team_type(team_id):
    if not isinstance(team_id, str):
        return "OTHER"
    up = team_id.upper()
    for key in ("OFAT", "FRAC", "CCD"):
        if key in up:
            return key
    return "OTHER"


def _find(name):
    for candidate in [Path(name), Path("/content") / name, Path.cwd() / name]:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"could not find {name} — did you upload it?")


def load_storefront(xlsx_path):
    df = pd.read_excel(xlsx_path, sheet_name="storefront")
    df = df.dropna(subset=["team_id"])
    df = df[df["infeasible"].fillna("").astype(str).str.strip() == ""]
    df["team_type"] = df["team_id"].apply(team_type)
    return df.reset_index(drop=True)

## Load the storefront

Point this at the file you just uploaded. Default is `formulation-storefront.xlsx`; change to `formulation-storefront-DEMO.xlsx` if you want to test the plots on the pre-simulated demo file.

In [ ]:
XLSX = "formulation-storefront.xlsx"   # or "formulation-storefront-DEMO.xlsx"
df = load_storefront(_find(XLSX))
print(f"Loaded {len(df)} feasible runs across {df['team_id'].nunique()} teams.")
df.head()

## Plot 1 — tensile trajectories, round 1

In [ ]:
def plot_tensile_round1(df, out_path):
    d = df[df["round"] == 1]
    fig, ax = plt.subplots(figsize=(10, 5), dpi=140)
    for team_id, grp in d.groupby("team_id"):
        color = TEAM_COLORS[team_type(team_id)]
        ax.plot(range(len(grp)), grp["tensile_mpa"].values, marker="o",
                label=team_id, color=color, alpha=0.85)
    ax.set_xlabel("run index within team")
    ax.set_ylabel("tensile (MPa)")
    ax.set_title("Round 1 — tensile by team")
    ax.legend(loc="lower right", fontsize=9)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(out_path)
    plt.show()

plot_tensile_round1(df, "reveal-01-tensile-round1.png")

## Plot 2 — elongation over the full game

In [ ]:
def plot_elongation_round2(df, out_path):
    fig, ax = plt.subplots(figsize=(10, 5), dpi=140)
    for team_id, grp in df.groupby("team_id"):
        color = TEAM_COLORS[team_type(team_id)]
        ax.plot(range(len(grp)), grp["elongation_pct"].values, marker="o",
                label=team_id, color=color, alpha=0.85)
        r2_start = (grp["round"] == 2).idxmax() if (grp["round"] == 2).any() else None
        if r2_start is not None and r2_start > 0:
            local_idx = grp.index.get_loc(r2_start)
            ax.axvline(local_idx - 0.5, color=color, ls=":", alpha=0.3)
    ax.set_xlabel("run index within team (round 1 → round 2)")
    ax.set_ylabel("elongation (%)")
    ax.set_title("Full game — elongation by team")
    ax.legend(loc="lower right", fontsize=9)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(out_path)
    plt.show()

plot_elongation_round2(df, "reveal-02-elongation-full-game.png")

## Plot 3 — Pareto scatter (tensile vs elongation)

**Suggested lead-off plot for the debrief.** Everyone maximized tensile in round 1; the round-2 elongation reveal separates the room.

In [ ]:
def plot_pareto(df, out_path):
    fig, ax = plt.subplots(figsize=(8, 6), dpi=140)
    for tt, grp in df.groupby("team_type"):
        ax.scatter(grp["tensile_mpa"], grp["elongation_pct"],
                   color=TEAM_COLORS[tt], label=tt, s=45, edgecolor="white",
                   linewidths=0.5, alpha=0.85)
    ax.axvline(6.0, color="k", ls="--", alpha=0.4, label="customer tensile floor")
    ax.set_xlabel("tensile (MPa)")
    ax.set_ylabel("elongation (%)")
    ax.set_title("Every feasible run, colored by team archetype")
    ax.legend(loc="lower left", fontsize=9)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(out_path)
    plt.show()

plot_pareto(df, "reveal-03-pareto.png")

## Plot 4 — best-observed responses per team

In [ ]:
def plot_summary(df, out_path):
    summary = (df.groupby("team_id")
                 .agg(best_tensile=("tensile_mpa", "max"),
                      best_elongation=("elongation_pct", "max"),
                      team_type=("team_type", "first"))
                 .reset_index())
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), dpi=140)
    for ax, col, ylabel in [(ax1, "best_tensile", "best tensile seen (MPa)"),
                            (ax2, "best_elongation", "best elongation seen (%)")]:
        colors = [TEAM_COLORS[t] for t in summary["team_type"]]
        ax.bar(summary["team_id"], summary[col], color=colors, edgecolor="white")
        ax.set_ylabel(ylabel)
        ax.tick_params(axis='x', rotation=30)
        ax.grid(alpha=0.3, axis="y")
    fig.suptitle("Best-observed responses per team", y=1.02)
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches="tight")
    plt.show()

plot_summary(df, "reveal-04-team-summary.png")

## Plot 5 — the true elongation surface

The ground-truth elongation surface as a function of coded filler (B) and crosslinker (C), with each team's queried points overlaid. **The single most powerful reveal plot** — students see exactly where their design let them sample.

In [ ]:
def plot_true_surface(df, out_path):
    n = 60
    Bg, Cg = np.meshgrid(np.linspace(-1, 1, n), np.linspace(-1, 1, n))
    Z = np.zeros_like(Bg)
    for i in range(n):
        for j in range(n):
            b, c = Bg[i, j], Cg[i, j]
            Z[i, j] = (ELONGATION["intercept"]
                      + ELONGATION["A_linear"] * 0
                      + ELONGATION["E_linear"] * 0
                      + ELONGATION["B_curvature"] * (b - ELONGATION["B_center"])**2
                      + ELONGATION["C_curvature"] * (c - ELONGATION["C_center"])**2
                      + ELONGATION["D_curvature"] * (0.5 - ELONGATION["D_center"])**2
                      + ELONGATION["BC"] * b * c)

    fig, ax = plt.subplots(figsize=(9, 7), dpi=140)
    cs = ax.contourf(Bg, Cg, Z, levels=20, cmap="viridis", alpha=0.7)
    fig.colorbar(cs, ax=ax, label="true mean elongation (%) at D=+0.5, A=E=0")

    ax.plot([ELONGATION["B_center"]], [ELONGATION["C_center"]], "w*",
            markersize=20, markeredgecolor="black", label="true elongation peak")

    for tt, grp in df.groupby("team_type"):
        mid_b = 0.5 * (RANGES["filler_phr"].low + RANGES["filler_phr"].high)
        half_b = 0.5 * (RANGES["filler_phr"].high - RANGES["filler_phr"].low)
        B_coded = (grp["filler_phr"] - mid_b) / half_b
        mid_c = 0.5 * (RANGES["crosslinker_phr"].low + RANGES["crosslinker_phr"].high)
        half_c = 0.5 * (RANGES["crosslinker_phr"].high - RANGES["crosslinker_phr"].low)
        C_coded = (grp["crosslinker_phr"] - mid_c) / half_c
        ax.scatter(B_coded, C_coded, color=TEAM_COLORS[tt], label=tt, s=55,
                   edgecolor="white", linewidths=0.8, alpha=0.9)

    ax.set_xlabel("filler_phr (coded)")
    ax.set_ylabel("crosslinker_phr (coded)")
    ax.set_title("True elongation surface — where did each team sample?")
    ax.legend(loc="lower left", fontsize=9)
    ax.grid(alpha=0.3, color="white", linestyle=":")
    ax.set_xlim(-1.1, 1.1)
    ax.set_ylim(-1.1, 1.1)
    fig.tight_layout()
    fig.savefig(out_path)
    plt.show()

plot_true_surface(df, "reveal-05-true-surface.png")

## Done

All five plots saved to the current folder. Download them from Colab's Files panel and project them in the debrief.

**Suggested narration order:** 3 → 5 → 1 → 2 → 4. See the facilitator run-of-show doc for line-by-line commentary.